# Industry-aware emotion classification

Consolidated from `yusra/Copy_of_emotion_with_RAG.ipynb`. See the root README for installation and store preparation.

The shared package owns prompts, schemas, retrieval, and response validation. Execute cells deliberately: API calls consume account quota. Store creation is a separate CLI command.

This legacy taxonomy covers only Health Insurance and Digital Health. The supplied shortlist is a demonstration input, not a reviewed classification. To use the original vector-retrieval path, set `industries=None` in the run cell and supply your industry store ID.

## 1. Load configuration

Install the project into your notebook kernel first. No client or remote store is created merely by opening the notebook.

In [ ]:
import json
from pathlib import Path

from orcalytics_experiments import run_experiment
from orcalytics_experiments.config import create_client, load_environment, store_id_for
from orcalytics_experiments.taxonomy import load_json

ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()), None
)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the repository checkout.")
load_environment(ROOT / ".env")
client = create_client()

## 2. Read the sample and taxonomy

Edit these paths to run your own article. All data paths are relative to the repository.

In [ ]:
article = (ROOT / "data/samples/hca_healthcare.txt").read_text(encoding="utf-8")
taxonomy = load_json(ROOT / "data/taxonomies/emotions_healthcare.json")
industries = load_json(ROOT / "data/samples/hca_industries.json")

## 3. Run the experiment

The JSON schema and taxonomy grounding are checked before a result is accepted. The `model` argument can override the original experiment default.

In [ ]:
result = run_experiment(
    client,
    "emotion",
    article,
    taxonomy,
    industries=industries,
    vector_store_id=store_id_for("emotion"),
    strategy="vector",
)
print(json.dumps(result["data"], indent=2, ensure_ascii=False))

## 4. Inspect provenance and token usage

A missing response ID means there were no supported downstream pairs to classify. Unsupported pairs are reported rather than fabricated.

In [ ]:
print(
    json.dumps(
        {
            key: result[key]
            for key in ("model", "strategy", "response_id", "usage", "unsupported_industries")
        },
        indent=2,
    )
)